# 02. Pose Baseline

## Goal

U3에서는 모델 결과를 보기 전에 다섯 이벤트 프레임을 기록하고, U5에서는 모든 Pose 후보를 저장한 뒤 주동작 선수와 한쪽 신체면을 직접 확인합니다. 모델 재실행 없이 선수 선택과 각도 결과를 다시 만들 수 있습니다.

## 현재 범위

먼저 U3 수동 이벤트를 확정한 뒤 U5 Pose 구역을 실행합니다. U5는 자세 점수나 정답 판정을 만들지 않고, 모든 사람 후보·선택 선수·confidence·관절 각도를 분리해 확인합니다.

## Setup

프로젝트 루트 또는 `notebooks/` 디렉터리에서 실행할 수 있습니다. 저장과 검증 로직은 `phases.py`, 프레임 읽기는 `video.py`를 사용합니다.

In [ ]:
from pathlib import Path
import os

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from snatch_analysis.config import ARTIFACT_ROOT, B00_VIDEO_ID
from snatch_analysis.phases import EVENT_NAMES, prepare_phase_annotations
from snatch_analysis.video import read_video_frame, read_video_properties


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(
        "프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요."
    )

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## Steps

### 1. 경로와 저장 설정

`OVERWRITE_PHASE_EVENTS=False`가 기본값입니다. 이미 B00 수동 이벤트가 있으면 사용자가 편집한 원본을 그대로 재사용합니다. 다섯 프레임을 다시 확정했을 때만 `True`로 바꾸세요.

In [ ]:
VIDEO_ID = B00_VIDEO_ID
MANIFEST_PATH = Path("data/metadata/videos.csv")
ANNOTATION_PATH = Path("data/annotations/phase_events.csv")
ARTIFACT_DIRECTORY = ARTIFACT_ROOT / VIDEO_ID
PREVIEW_FRAME_INDEX = 0
OVERWRITE_PHASE_EVENTS = False
ANNOTATOR = ""
NOTES = ""

### 2. B00 영상 기준 확인

U1 Manifest와 실제 영상 속성이 일치하는지 다시 확인합니다. 이 단계에서 확인한 `frame_count`가 이벤트 범위와 `phase_by_frame.csv` 행 수의 기준입니다.

In [ ]:
if VIDEO_ID != B00_VIDEO_ID:
    raise ValueError(f"U3는 {B00_VIDEO_ID}만 지원합니다: {VIDEO_ID}")
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest가 없습니다: {MANIFEST_PATH}. 01_data_diagnosis.ipynb를 먼저 실행하세요."
    )

manifest = pd.read_csv(MANIFEST_PATH)
selected_rows = manifest.loc[manifest["video_id"] == VIDEO_ID]
if len(selected_rows) != 1:
    raise ValueError(
        f"Manifest에서 VIDEO_ID={VIDEO_ID!r} 행이 정확히 하나여야 합니다: "
        f"{len(selected_rows)}개"
    )

selected_video = selected_rows.iloc[0]
video_path = Path(selected_video["path"])
frame_count = int(selected_video["frame_count"])
video_properties = read_video_properties(video_path)
if video_properties["read_status"] != "ok":
    raise RuntimeError("B00 영상 기준 실패: read_status_ok")

video_checks = {
    "frame_count_matches": video_properties["frame_count"] == frame_count,
    "fps_matches": abs(
        float(video_properties["fps"]) - float(selected_video["fps"])
    ) < 1e-6,
    "size_matches": (
        video_properties["width"],
        video_properties["height"],
    )
    == (int(selected_video["width"]), int(selected_video["height"])),
}
failed_video_checks = [
    name for name, passed in video_checks.items() if not passed
]
if failed_video_checks:
    raise RuntimeError(
        f"B00 영상 기준 실패: {', '.join(failed_video_checks)}"
    )

pd.Series(
    {
        "video_id": VIDEO_ID,
        "FPS": video_properties["fps"],
        "frame_range": f"0 ~ {frame_count - 1}",
        "frame_count": frame_count,
        "decoded_size": (
            video_properties["height"],
            video_properties["width"],
            3,
        ),
        "contract": "PASS",
    },
    name="value",
).to_frame()

### 3. 원본 프레임을 보며 이벤트 탐색

`PREVIEW_FRAME_INDEX`를 바꾸고 이 셀만 다시 실행하면서 동작 전환점을 찾으세요. Pose 점이나 원판 bbox는 표시하지 않습니다. OpenCV BGR 프레임은 화면에 표시할 때만 RGB로 변환합니다.

In [ ]:
preview_frame = read_video_frame(video_path, PREVIEW_FRAME_INDEX)
preview_rgb = cv2.cvtColor(preview_frame, cv2.COLOR_BGR2RGB)
preview_time_sec = PREVIEW_FRAME_INDEX / float(video_properties["fps"])

plt.figure(figsize=(6, 8))
plt.imshow(preview_rgb)
plt.title(
    f"{VIDEO_ID} | frame {PREVIEW_FRAME_INDEX} | {preview_time_sec:.3f}s"
)
plt.axis("off")
plt.show()

### 4. 다섯 이벤트 프레임 입력

프레임을 충분히 확인한 뒤 `None`을 정수 frame index로 바꾸세요. 조건은 `bar_off_floor < knee_pass < max_extension < catch_stable < recovery_stable`이며 모두 `0..frame_count-1` 안에 있어야 합니다. 기존 원본을 재사용할 때는 기본값을 그대로 두어도 됩니다.

In [ ]:
EVENT_FRAMES = {
    "bar_off_floor": None,
    "knee_pass": None,
    "max_extension": None,
    "catch_stable": None,
    "recovery_stable": None,
}

### 5. 검증하고 로컬 CSV 저장

처음 저장할 때는 다섯 값이 모두 필요합니다. 원본 `data/annotations/phase_events.csv`는 기본적으로 재사용하며, 현재 실행에 검증된 B00 행과 모든 프레임 구간은 ignored `artifacts/b00_sn_f_01/`에 snapshot으로 저장합니다.

In [ ]:
phase_events, phase_by_frame, source_action = prepare_phase_annotations(
    video_id=VIDEO_ID,
    event_frames=EVENT_FRAMES,
    frame_count=frame_count,
    annotation_path=ANNOTATION_PATH,
    artifact_directory=ARTIFACT_DIRECTORY,
    annotator=ANNOTATOR,
    notes=NOTES,
    overwrite=OVERWRITE_PHASE_EVENTS,
)

In [ ]:
pd.Series(
    {
        "source_action": source_action,
        "event_rows": len(phase_events),
        "phase_rows": len(phase_by_frame),
        "source_path": ANNOTATION_PATH.as_posix(),
        "snapshot_path": (ARTIFACT_DIRECTORY / "phase_by_frame.csv").as_posix(),
        "manual_checkpoint": "PASS",
    },
    name="value",
).to_frame()

## Checks

### 6. 저장된 이벤트와 구간 수 확인

이벤트 5행과 모든 프레임의 구간이 생성됐는지 확인합니다. `recovery_complete=True`는 `recovery_stable` 프레임부터 표시됩니다.

In [ ]:
phase_events

In [ ]:
phase_order = [
    "Set-up",
    "First Pull",
    "Second Pull",
    "Drop Under & Catch",
    "Recovery",
]
phase_by_frame["phase"].value_counts().reindex(
    phase_order, fill_value=0
).rename("frame_count").to_frame()

In [ ]:
event_index_set = set(phase_events["frame_index"].astype(int))
boundary_indices = sorted(
    {0, frame_count - 1}
    | event_index_set
    | {index - 1 for index in event_index_set if index > 0}
)
phase_by_frame.loc[
    phase_by_frame["frame_index"].isin(boundary_indices)
].reset_index(drop=True)

## 관찰 기록

- `bar_off_floor`을 선택한 근거:
- `knee_pass`를 선택한 근거:
- `max_extension`을 선택한 근거:
- `catch_stable`을 선택한 근거:
- `recovery_stable`을 선택한 근거:
- 판단이 어려웠던 경계와 후보 프레임:

## U5. Pose 후보와 주동작 선수

아래 셀은 U3의 `manual_checkpoint=PASS` 이후 실행합니다. 모든 사람 후보는 진단용 CSV로 보존하고, 자동 선택된 주동작 선수 한 명만 관절·각도 계산에 사용합니다. 자동 기준이 충돌할 때만 수동 override를 입력합니다.

In [ ]:
import json
import time

import numpy as np

from snatch_analysis.config import POSE_MODEL_NAME
from snatch_analysis.pose import (
    COCO_KEYPOINT_NAMES,
    POSE_CANDIDATE_COLUMNS,
    calculate_joint_angles,
    pose_result_to_candidates,
    resolve_initial_primary_lifter,
    select_initial_primary_lifter,
    resolve_inference_device,
    save_pose_candidate_cache,
    save_selected_pose_keypoints,
    select_primary_lifter_track,
    summarize_body_side_confidence,
    validate_pose_candidate_cache,
    verify_ultralytics_safe_load,
)
from snatch_analysis.video import iter_video_frames


def load_run_summary():
    return (
        json.loads(RUN_SUMMARY_PATH.read_text(encoding="utf-8"))
        if RUN_SUMMARY_PATH.exists()
        else {}
    )


def save_run_summary(summary):
    RUN_SUMMARY_PATH.write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )

### 7. Pose 추론 설정

처음 한 번만 `REINFER_POSE=True`로 바꾸세요. 이후에는 기본값 `False`로 `pose_candidates.csv`를 재사용합니다. 현재 영상·모델·입력 크기·bbox confidence와 일치하는 완료 cache만 재사용합니다. `POSE_CONFIDENCE`는 사람 bbox 탐지 기준이고 `KEYPOINT_CONFIDENCE_THRESHOLD`는 선택된 관절을 각도 계산에 사용할 기준입니다. 한 실험에서는 한 값만 바꾸세요.

In [ ]:
POSE_CANDIDATE_PATH = ARTIFACT_DIRECTORY / "pose_candidates.csv"
POSE_KEYPOINT_PATH = ARTIFACT_DIRECTORY / "pose_keypoints.csv"
JOINT_ANGLE_PATH = ARTIFACT_DIRECTORY / "joint_angles.csv"
RUN_SUMMARY_PATH = ARTIFACT_DIRECTORY / "run_summary.json"

REINFER_POSE = False
REQUESTED_DEVICE = "mps"
POSE_IMAGE_SIZE = 640
POSE_CONFIDENCE = 0.25
KEYPOINT_CONFIDENCE_THRESHOLD = 0.50

### 8. 전체 frame 후보 추론 또는 저장 결과 재사용

추론은 frame을 한 장씩 읽습니다. Ultralytics import 전에 restricted checkpoint loading을 검증하며, 지원되지 않으면 일반 pickle 방식으로 우회하지 않고 중단합니다. MPS를 요청하고 사용할 수 없을 때만 CPU로 대체합니다.

In [ ]:
ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)
pose_elapsed_seconds = None
pose_device = None
expected_pose_run = {
    "status": "complete",
    "video_id": VIDEO_ID,
    "frame_count": frame_count,
    "processed_frame_count": frame_count,
    "model": POSE_MODEL_NAME,
    "image_size": POSE_IMAGE_SIZE,
    "bbox_confidence_threshold": POSE_CONFIDENCE,
}

if REINFER_POSE:
    safe_load = verify_ultralytics_safe_load()
    from ultralytics import YOLO

    pose_device = resolve_inference_device(REQUESTED_DEVICE)
    pose_model = YOLO(POSE_MODEL_NAME)
    candidate_tables = []
    started_at = time.perf_counter()
    processed_frame_count = 0
    for frame_index, timestamp_seconds, frame in iter_video_frames(
        video_path, expected_frame_count=frame_count
    ):
        result = pose_model.predict(
            source=frame,
            imgsz=POSE_IMAGE_SIZE,
            conf=POSE_CONFIDENCE,
            device=pose_device["actual_device"],
            verbose=False,
        )[0]
        candidate_tables.append(
            pose_result_to_candidates(
                result, VIDEO_ID, frame_index, timestamp_seconds
            )
        )
        processed_frame_count = frame_index + 1
    pose_elapsed_seconds = time.perf_counter() - started_at
    pose_candidates = (
        pd.concat(candidate_tables, ignore_index=True)
        if candidate_tables
        else pd.DataFrame(columns=POSE_CANDIDATE_COLUMNS)
    )
    pose_run = {
        **expected_pose_run,
        "processed_frame_count": processed_frame_count,
        "requested_device": pose_device["requested_device"],
        "actual_device": pose_device["actual_device"],
        "fallback_reason": pose_device["fallback_reason"],
        "safe_load": safe_load,
        "elapsed_seconds": round(pose_elapsed_seconds, 3),
    }
    validate_pose_candidate_cache(
        pose_candidates,
        expected_video_id=VIDEO_ID,
        expected_frame_count=frame_count,
        run_metadata=pose_run,
        expected_run_metadata=expected_pose_run,
    )
    save_pose_candidate_cache(pose_candidates, POSE_CANDIDATE_PATH)
    run_summary = load_run_summary()
    run_summary["pose"] = pose_run
    save_run_summary(run_summary)
    pose_source_action = "inferred"
elif POSE_CANDIDATE_PATH.exists():
    pose_candidates = pd.read_csv(POSE_CANDIDATE_PATH)
    run_summary = load_run_summary()
    try:
        validate_pose_candidate_cache(
            pose_candidates,
            expected_video_id=VIDEO_ID,
            expected_frame_count=frame_count,
            run_metadata=run_summary.get("pose", {}),
            expected_run_metadata=expected_pose_run,
        )
    except ValueError as error:
        raise RuntimeError(
            f"Pose cache를 재사용할 수 없습니다: {error} "
            "REINFER_POSE=True로 다시 실행하세요."
        ) from error
    pose_source_action = "reused"
else:
    pose_candidates = pd.DataFrame(columns=POSE_CANDIDATE_COLUMNS)
    pose_source_action = "not_run"
    print("Pose 후보가 없습니다. 처음 한 번 REINFER_POSE=True로 실행하세요.")

In [ ]:
pd.Series(
    {
        "source_action": pose_source_action,
        "candidate_rows": len(pose_candidates),
        "candidate_frames": pose_candidates["frame_index"].nunique(),
        "detected_people": len(
            pose_candidates[["frame_index", "candidate_id"]].drop_duplicates()
        ),
        "elapsed_seconds": pose_elapsed_seconds,
        "actual_device": (pose_device or {}).get("actual_device"),
        "candidate_path": POSE_CANDIDATE_PATH.as_posix(),
    },
    name="value",
).to_frame()

### 9. 첫 유효 frame에서 주동작 선수 자동 선택 확인

주동작 선수는 카메라에 가장 가까워 bbox가 가장 크고 가로 중앙에 있다는 촬영 조건을 사용합니다. 두 기준이 같은 후보를 가리킬 때만 자동 선택하고, 기준 충돌·동률은 `confirmation_required`로 남깁니다. 전체 후보 bbox는 진단용이며 관절·각도에는 선택 track 한 명만 사용합니다.

In [ ]:
if pose_candidates.empty:
    primary_selection = {
        "frame_index": None,
        "candidate_id": None,
        "status": "not_available",
        "reason": "no_pose_candidates",
    }
    first_candidate_boxes = pd.DataFrame()
else:
    primary_selection = select_initial_primary_lifter(
        pose_candidates,
        frame_width=int(video_properties["width"]),
    )
    first_candidate_frame = int(primary_selection["frame_index"])
    first_candidate_boxes = (
        pose_candidates.loc[
            pose_candidates["frame_index"] == first_candidate_frame,
            [
                "candidate_id", "bbox_confidence", "bbox_x1",
                "bbox_y1", "bbox_x2", "bbox_y2",
            ],
        ]
        .drop_duplicates("candidate_id")
        .sort_values("candidate_id")
    )
    candidate_preview = read_video_frame(video_path, first_candidate_frame).copy()
    for candidate in first_candidate_boxes.itertuples():
        x1, y1, x2, y2 = map(
            int, (candidate.bbox_x1, candidate.bbox_y1, candidate.bbox_x2, candidate.bbox_y2)
        )
        cv2.rectangle(candidate_preview, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            candidate_preview,
            f"id={candidate.candidate_id} conf={candidate.bbox_confidence:.2f}",
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2, cv2.LINE_AA,
        )
    plt.figure(figsize=(6, 8))
    plt.imshow(cv2.cvtColor(candidate_preview, cv2.COLOR_BGR2RGB))
    plt.title(f"first valid pose frame {first_candidate_frame}")
    plt.axis("off")
    plt.show()

print(primary_selection)
first_candidate_boxes

### 10. 주동작 선수 자동 선택·예외 override

기본값은 자동 선택입니다. 첫 유효 frame에서 가장 큰 bbox와 가로 중앙에 가장 가까운 bbox가 같은 후보면 바로 추적합니다. 두 기준이 다르거나 동률이면 `MANUAL_OVERRIDE_CANDIDATE_ID`에 preview의 ID를 입력하세요.

In [ ]:
MANUAL_OVERRIDE_CANDIDATE_ID = None

In [ ]:
pose_keypoints_preview = pd.DataFrame()
initial_selection = resolve_initial_primary_lifter(
    primary_selection, MANUAL_OVERRIDE_CANDIDATE_ID
)
initial_candidate_id = initial_selection["candidate_id"]
selection_method = initial_selection["selection_method"]
selection_summary = {
    "status": (
        "not_available" if pose_candidates.empty
        else "in_progress" if initial_candidate_id is not None
        else primary_selection["status"]
    ),
    "reason": primary_selection["reason"],
    "selection_method": selection_method,
    "initial_frame_index": primary_selection["frame_index"],
    "initial_candidate_id": initial_candidate_id,
    "keypoint_confidence_threshold": KEYPOINT_CONFIDENCE_THRESHOLD,
}
run_summary = load_run_summary()
run_summary["pose_selection"] = selection_summary
save_run_summary(run_summary)
JOINT_ANGLE_PATH.unlink(missing_ok=True)

if pose_candidates.empty:
    POSE_KEYPOINT_PATH.unlink(missing_ok=True)
    print("Pose 후보가 없어 선수 track을 만들지 않았습니다.")
elif initial_candidate_id is None:
    POSE_KEYPOINT_PATH.unlink(missing_ok=True)
    print("자동 선택 기준이 충돌했습니다. MANUAL_OVERRIDE_CANDIDATE_ID를 입력하세요.")
else:
    try:
        pose_keypoints_preview = select_primary_lifter_track(
            pose_candidates,
            frame_count=frame_count,
            initial_frame_index=int(primary_selection["frame_index"]),
            initial_candidate_id=int(initial_candidate_id),
            keypoint_confidence_threshold=KEYPOINT_CONFIDENCE_THRESHOLD,
            fps=float(video_properties["fps"]),
        )
        save_selected_pose_keypoints(pose_keypoints_preview, POSE_KEYPOINT_PATH)
    except Exception as error:
        run_summary = load_run_summary()
        run_summary["pose_selection"] = {
            **selection_summary,
            "status": "failed",
            "failure_reason": f"{type(error).__name__}: {error}",
        }
        save_run_summary(run_summary)
        raise
    run_summary = load_run_summary()
    run_summary["pose_selection"] = {**selection_summary, "status": "complete"}
    save_run_summary(run_summary)
    print(f"{selection_method} 선택 track 저장: {POSE_KEYPOINT_PATH}")

pose_keypoints_preview.head(17)

#### 선택 track 연속성 확인

U3의 다섯 이벤트와 bbox 변화가 큰 frame을 함께 봅니다. `candidate_id`는 frame마다 다시 매겨져 바뀔 수 있으므로 숫자가 아니라 녹색 bbox 속 인물이 같은 주동작 선수인지 확인하세요. 잘못 선택된 경우에만 위 셀의 수동 override를 사용합니다.

In [ ]:
track_boxes = pd.DataFrame()
review_frame_indices = []
if not pose_keypoints_preview.empty:
    track_boxes = (
        pose_keypoints_preview[[
            "frame_index", "candidate_id", "bbox_x1", "bbox_y1",
            "bbox_x2", "bbox_y2", "bbox_area",
            "bbox_center_x", "bbox_center_y",
        ]]
        .drop_duplicates("frame_index")
        .sort_values("frame_index")
    )
    valid_boxes = track_boxes.dropna(subset=["candidate_id"]).copy()
    if not valid_boxes.empty:
        valid_boxes["motion_score"] = (
            np.hypot(
                valid_boxes["bbox_center_x"].diff(),
                valid_boxes["bbox_center_y"].diff(),
            )
            / valid_boxes["bbox_area"].pow(0.5).shift().clip(lower=1.0)
            + np.log(valid_boxes["bbox_area"].clip(lower=1.0)).diff().abs()
        )
        motion_frames = valid_boxes.nlargest(3, "motion_score")["frame_index"]
    else:
        motion_frames = pd.Series(dtype=int)
    review_frame_indices = sorted(
        set(phase_events["frame_index"].astype(int))
        | set(motion_frames.dropna().astype(int))
    )

if review_frame_indices:
    column_count = 3
    row_count = (len(review_frame_indices) + column_count - 1) // column_count
    figure, axes = plt.subplots(row_count, column_count, figsize=(15, 4 * row_count))
    for axis in np.asarray(axes).reshape(-1):
        axis.axis("off")
    for axis, frame_index in zip(
        np.asarray(axes).reshape(-1), review_frame_indices, strict=False
    ):
        review_frame = read_video_frame(video_path, frame_index).copy()
        selected_box = track_boxes.loc[track_boxes["frame_index"] == frame_index]
        label = "N/A"
        if not selected_box.empty and pd.notna(selected_box.iloc[0]["candidate_id"]):
            box = selected_box.iloc[0]
            x1, y1, x2, y2 = map(
                int, (box.bbox_x1, box.bbox_y1, box.bbox_x2, box.bbox_y2)
            )
            cv2.rectangle(review_frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
            label = f"candidate {int(box.candidate_id)}"
        axis.imshow(cv2.cvtColor(review_frame, cv2.COLOR_BGR2RGB))
        axis.set_title(f"frame {frame_index} | {label}")
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("선택 track이 없어 연속성을 확인할 수 없습니다.")

In [ ]:
run_summary.get("pose_selection", {})

### 11. 좌우 confidence를 본 뒤 신체면 고정

각도에 쓰는 어깨·팔꿈치·손목·엉덩이·무릎·발목의 영상 전체 confidence와 사용 가능 비율입니다. 더 잘 보이는 한쪽을 선택하되, frame마다 좌우를 바꾸지 않습니다.

In [ ]:
body_side_summary = (
    summarize_body_side_confidence(pose_keypoints_preview)
    if not pose_keypoints_preview.empty
    else pd.DataFrame()
)
body_side_summary

In [ ]:
BODY_SIDE = None  # "left" 또는 "right"

In [ ]:
joint_angles = pd.DataFrame()
if pose_keypoints_preview.empty:
    POSE_KEYPOINT_PATH.unlink(missing_ok=True)
    JOINT_ANGLE_PATH.unlink(missing_ok=True)
    print("선택된 선수 track이 없어 각도를 저장하지 않습니다.")
elif BODY_SIDE is None:
    JOINT_ANGLE_PATH.unlink(missing_ok=True)
    run_summary = load_run_summary()
    pose_selection = run_summary.setdefault("pose_selection", {})
    pose_selection.pop("body_side", None)
    pose_selection["angle_status"] = "not_run"
    save_run_summary(run_summary)
    print("confidence 표를 확인하고 BODY_SIDE를 선택하세요.")
else:
    joint_angles = calculate_joint_angles(pose_keypoints_preview, BODY_SIDE)
    joint_angles.to_csv(JOINT_ANGLE_PATH, index=False)
    run_summary = load_run_summary()
    run_summary.setdefault("pose_selection", {})["body_side"] = BODY_SIDE
    run_summary["pose_selection"]["angle_status"] = "complete"
    save_run_summary(run_summary)
    print(f"고정 신체면: {BODY_SIDE} | 각도 저장: {JOINT_ANGLE_PATH}")

joint_angles.head()

### 12. Confidence 분포와 최악/N/A frame 확인

평균만 보지 않고 N/A 관절이 많은 frame과 낮은 confidence frame을 먼저 봅니다. 좌표는 보간하지 않았으므로 `reason`이 실패 원인을 그대로 보여줍니다.

In [ ]:
if pose_keypoints_preview.empty:
    confidence_by_side = pd.DataFrame()
    worst_pose_frames = pd.DataFrame()
    not_available_joints = pd.DataFrame()
else:
    confidence_by_side = pose_keypoints_preview.loc[
        pose_keypoints_preview["joint_name"].isin(COCO_KEYPOINT_NAMES[5:])
    ].copy()
    confidence_by_side["side"] = confidence_by_side["joint_name"].str.split("_").str[0]
    for side, values in confidence_by_side.groupby("side")["confidence"]:
        plt.hist(values.dropna(), bins=20, alpha=0.5, label=side)
    plt.title("Angle-joint confidence distribution")
    plt.xlabel("confidence")
    plt.ylabel("joint rows")
    plt.legend()
    plt.show()

    worst_pose_frames = (
        pose_keypoints_preview.groupby("frame_index")
        .agg(
            mean_confidence=("confidence", "mean"),
            not_available_joints=("status", lambda values: (values != "ok").sum()),
        )
        .sort_values(["not_available_joints", "mean_confidence"], ascending=[False, True])
        .head(20)
        .reset_index()
    )
    not_available_joints = pose_keypoints_preview.loc[
        pose_keypoints_preview["status"] != "ok",
        ["frame_index", "joint_name", "confidence", "reason"],
    ].head(30)

worst_pose_frames

In [ ]:
not_available_joints

## Next Steps

자동 선택된 주동작 선수 track과 고정 신체면을 확인하고 `pose_keypoints.csv`, `joint_angles.csv`가 각각 254 frame 기준으로 저장되면 U6 원판 frame 추출·라벨 검증으로 넘어갑니다. 자동 선택이 틀린 실패 사례에서만 수동 override를 사용합니다. `yolov8s-pose.pt` 비교와 자동 자세 평가는 아직 진행하지 않습니다.